In [ ]:
from pathlib import Path
import pandas as pd

# Display settings (optional)
pd.set_option('display.max_colwidth', 120)

# How many rows to show when displaying DataFrames in the notebook
pd.set_option('display.max_rows', 200)  # increase if you want more
pd.set_option('display.min_rows', 50)

In [5]:
# Choose the input JSON file (JSON Whole Model export)
file_name = "ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json"

# Default: workspace-root/JSON Whole Model/<file_name> (works when notebook is in PyDataTransform/)
input_path = (Path('..') / 'JSON Whole Model' / file_name).resolve()
if not input_path.exists():
    input_path = (Path('JSON Whole Model') / file_name).resolve()

assert input_path.exists(), f"File not found: {input_path}"

df = pd.read_json(input_path)
df.shape

(180, 4)

#### Model element Name counts
This notebook loads a `JSON Whole Model/*.json` export and prints a table with **Name**, **DbId**, **GUID**, and the **total count of that Name** across the model (duplicates included).

In [ ]:
def _extract_guid(props):
    # `props` is the element's `Properties` array.
    # We look for the entry with displayName == 'GUID' (case-insensitive).
    if not isinstance(props, list):
        return None
    for item in props:
        if not isinstance(item, dict):
            continue
        display_name = str(item.get('displayName', '')).strip()
        if display_name.lower() == 'guid':
            return item.get('value')
    return None

# Build GUID + NameCount columns
df['GUID'] = df['Properties'].apply(_extract_guid)
name_counts = df['Name'].value_counts(dropna=False)
df['NameCount'] = df['Name'].map(name_counts)

table = (
    df[['Name', 'DbId', 'GUID', 'NameCount']]
    .sort_values(['Name', 'DbId'], kind='stable')
    .reset_index(drop=True)
 )

# Show more rows explicitly (independent of pandas display.max_rows)
rows_to_show = 200  # set to None to show all rows (can be slow/huge)
table if rows_to_show is None else table.head(rows_to_show)

,Name,DbId,GUID,NameCount
0,1JNL9322340_A-1JNL9322340 - Pyramid,5,8d3567ff-612b-3878-a628-4b9d838df628,1
1,"1JNL9362899_A-Electrical design requirements, MVS1, NER, Aux Tx-building, Cable ways",6,cd06ec2a-dde8-35e7-bb84-94eaba646aaa,1
2,1JNL9441111_A-Cable Ladder 90 450,14,c9232a6d-75c5-3739-a3c2-3777ac67e597,3
3,1JNL9441111_A-Cable Ladder 90 450,15,3d931300-8be8-3a23-a464-9cedb7d98460,3
4,1JNL9441111_A-Cable Ladder 90 450,19,19f22aef-af24-3a1c-b63f-1703e94e50ba,3
...,...,...,...,...
175,Body,180,None,161
176,Body,181,None,161
177,Body,182,None,161
178,Default Building,3,f73c5fbf-200c-394e-8dd7-fbddc1837410,1


In [ ]:
# Export the table
out_dir = input_path.parent
csv_path = out_dir / f"{input_path.stem}_name_table.csv"
xlsx_path = out_dir / f"{input_path.stem}_name_table.xlsx"

table.to_csv(csv_path, index=False, encoding='utf-8-sig')
print("Wrote CSV:", csv_path)

# Excel export requires openpyxl (recommended)
try:
    import openpyxl  # noqa: F401
    table.to_excel(xlsx_path, index=False)
    print("Wrote Excel:", xlsx_path)
except ImportError:
    print("Excel export skipped: package 'openpyxl' is not installed.")
    print("Run: pip install openpyxl  (or use notebook package install), then re-run this cell.")